In [1]:
import os
import re
from pathlib import Path
from typing import Union

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import yaml
from torch import Tensor

from torchaudio.compliance import kaldi
from torchaudio.transforms import ComputeDeltas

In [2]:
class AttrDict(dict):
    def __init__(self, *args, **kwargs):
        super(AttrDict, self).__init__(*args, **kwargs)

    def __getattr__(self, item):
        if item not in self:
            return None
        if type(self[item]) is dict:
            self[item] = AttrDict(self[item])
        return self[item]

    def __setattr__(self, item, value):
        self.__dict__[item] = value

In [3]:
def find_repo_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "config" / "config.yaml").exists():
            return candidate
    raise FileNotFoundError("Could not find config/config.yaml from the current working directory")


REPO_ROOT = find_repo_root()
CONFIG_PATH = REPO_ROOT / "config" / "config.yaml"

with open(CONFIG_PATH) as file:
    config = AttrDict(yaml.load(file, Loader=yaml.FullLoader))
config = config.data

TIMIT_ROOT = REPO_ROOT / config.name
RAW_ROOT = TIMIT_ROOT / config.raw_dir
FEATURES_ROOT = TIMIT_ROOT / config.features_dir

In [4]:
def preprocess_transcript(transcript):
    transcript = re.sub(r'[^\w\s]', '', transcript)
    transcript = transcript.lower()
    transcript = transcript.strip()
    transcript = transcript.replace('"', '')
    return transcript

In [5]:
def repo_relative(path: Union[str, Path]) -> str:
    return str(Path(path).resolve().relative_to(REPO_ROOT))


def repo_path(path: Union[str, Path]) -> Path:
    path = Path(path)
    return path if path.is_absolute() else REPO_ROOT / path


def feature_path_for_audio(audio_path: Union[str, Path]) -> str:
    audio_path = repo_path(audio_path)
    feature_rel = audio_path.resolve().relative_to(RAW_ROOT).with_suffix(".npy")
    return repo_relative(FEATURES_ROOT / feature_rel)


def load_audio(audio_path: Union[str, Path]):
    samples, sample_rate = sf.read(repo_path(audio_path), dtype="float32", always_2d=True)
    return torch.from_numpy(samples.T), sample_rate


def audio_num_frames(audio_path: Union[str, Path]) -> int:
    return sf.info(repo_path(audio_path)).frames


def load_transcripts(data_path, transcript_path):
    transcripts = []
    df = pd.read_csv(TIMIT_ROOT / data_path).dropna(subset=["index"])
    audio_paths = df.loc[df["path_from_data_dir"].str.endswith(".WAV"), "path_from_data_dir"]

    for audio_rel_path in audio_paths:
        audio_path = RAW_ROOT / audio_rel_path
        txt_path = audio_path.with_suffix(".TXT")

        with open(txt_path, "r") as f:
            transcript = f.read().split(" ", 2)[-1]
            duration = audio_num_frames(audio_path)
            transcript = preprocess_transcript(transcript)
            audio_path = repo_relative(audio_path)
            transcripts.append({
                "audio_path": audio_path,
                "feature_path": feature_path_for_audio(audio_path),
                "transcript": transcript,
                "duration": duration,
            })

    transcripts_df = pd.DataFrame(transcripts)
    transcripts_df.to_csv(TIMIT_ROOT / transcript_path, index=False)

In [6]:
train_data_path = config.train_data
core_train_path = config.core_train

test_data_path = config.test_data
core_test_path = config.core_test
core_val_path = config.core_val

In [7]:
def get_audio_features(
    audio_path: str,
    config: AttrDict,
    mean: Union[float, Tensor] = 0.0,
    std: Union[float, Tensor] = 1.0,
):
    x, sample_rate = load_audio(audio_path)
    mfcc = kaldi.mfcc(
        waveform=x,
        sample_frequency=sample_rate,
        frame_length=config.win_size / sample_rate * 1000,
        frame_shift=config.hop_size / sample_rate * 1000,
        window_type=config.window_type,
        num_ceps=config.num_ceps,
        num_mel_bins=config.num_mel_bins,
        preemphasis_coefficient=0.97,
        use_energy=True,
    )
    delta = ComputeDeltas()(mfcc.transpose(0, 1)).transpose(0, 1)
    features = torch.cat((mfcc, delta), dim=1)
    return (features - mean) / std

In [13]:
# Regenerate transcript CSVs from the raw manifests so this cell is rerunnable.
load_transcripts(train_data_path, core_train_path)
load_transcripts(test_data_path, core_test_path)

test_speakers = {
    "DAB0", "TAS1", "JMP0", "LLL0", "BPM0", "CMJ0", "GRT0", "JLN0", "WBT0",
    "WEW0", "LNT0", "TLS0", "KLT0", "JDH0", "NJM0", "PAM0", "ELC0", "PAS0",
    "PKT0", "JLM0", "NLP0", "MGD0", "DHC0", "MLD0"
}


def path_parts_from_row(row):
    return repo_path(row["audio_path"]).resolve().relative_to(RAW_ROOT).parts


def sentence_id(row):
    return Path(path_parts_from_row(row)[-1]).stem


def speaker_id(row):
    return path_parts_from_row(row)[2][1:]


def is_core_test_set(row):
    return speaker_id(row) in test_speakers and not sentence_id(row).startswith("SA")


def is_core_train_set(row):
    return not sentence_id(row).startswith("SA")


def finalize_audio_paths(df):
    df = df.copy()
    df["audio_path"] = df["audio_path"].map(lambda path: repo_relative(repo_path(path)))
    df["feature_path"] = df["audio_path"].map(feature_path_for_audio)
    return df


# Build final train/validation splits before computing normalization statistics.
df_train = pd.read_csv(TIMIT_ROOT / core_train_path)
df_train = df_train[df_train.apply(is_core_train_set, axis=1)].copy()
df_val = df_train.sample(n=184, random_state=42)
df_train = df_train.drop(df_val.index)

finalize_audio_paths(df_train).to_csv(TIMIT_ROOT / core_train_path, index=False)
finalize_audio_paths(df_val).to_csv(TIMIT_ROOT / core_val_path, index=False)

# Build final core test split. Path normalization keeps this cell rerunnable.
df_test = pd.read_csv(TIMIT_ROOT / core_test_path)
df_test = df_test[df_test.apply(is_core_test_set, axis=1)].copy()
finalize_audio_paths(df_test).to_csv(TIMIT_ROOT / core_test_path, index=False)

print(f"train dataset size: {len(df_train)}")
print(f"validation dataset size: {len(df_val)}")
print(f"test dataset size: {len(df_test)}")

features = []
for audio_path in df_train["audio_path"]:
    feature = get_audio_features(repo_path(audio_path), config)
    features.append(feature)

features_concat = torch.cat(features, dim=0)
global_train_mean = torch.mean(features_concat, axis=0, keepdims=True)
global_train_std = torch.std(features_concat, axis=0, keepdims=True)

print("TRAIN_MEAN\n", global_train_mean.squeeze())
print("TRAIN_STD\n", global_train_std.squeeze())

train dataset size: 3512
validation dataset size: 184
test dataset size: 192
TRAIN_MEAN
 tensor([ 1.1604e-02, -1.0365e+01, -1.1453e+01, -5.8834e+00, -1.6807e+01,
        -1.2180e+01, -9.7095e+00, -9.8986e+00, -1.2671e+00, -5.4334e+00,
        -1.6563e+00, -3.2267e+00, -2.9995e+00, -4.9530e-12,  2.4902e-03,
         2.7990e-03, -5.2212e-03, -4.5569e-03,  6.6362e-03, -9.6694e-04,
         1.8054e-03, -7.9193e-03, -6.9905e-03,  1.8198e-03, -6.0241e-03,
        -5.4133e-03])
TRAIN_STD
 tensor([ 0.1031, 20.1937, 15.3606, 17.6085, 18.4885, 17.2309, 17.0437, 17.4238,
        15.6127, 15.7126, 13.5232, 13.5087, 11.7996,  0.0234,  4.3929,  3.7379,
         3.6424,  4.1886,  4.0396,  4.0304,  4.2022,  3.9929,  3.9115,  3.5321,
         3.3752,  3.0935])


In [9]:
def save_audios_features(transcript_path: str, config: AttrDict, mean: Tensor, std: Tensor):
    df = pd.read_csv(TIMIT_ROOT / transcript_path)
    for _, row in df.iterrows():
        audio_path = repo_path(row["audio_path"])
        features = get_audio_features(audio_path, config, mean, std)
        features = features.unsqueeze(0)
        save_path = repo_path(row["feature_path"])
        save_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(save_path, features.numpy())
        print(repo_relative(save_path))

In [10]:
save_audios_features(core_train_path, config, global_train_mean, global_train_std)
save_audios_features(core_val_path, config, global_train_mean, global_train_std)
save_audios_features(core_test_path, config, global_train_mean, global_train_std)

timit/features/TRAIN/DR4/MMDM0/SI681.npy
timit/features/TRAIN/DR4/MMDM0/SX411.npy
timit/features/TRAIN/DR4/MMDM0/SX231.npy
timit/features/TRAIN/DR4/MMDM0/SX51.npy
timit/features/TRAIN/DR4/MMDM0/SX141.npy
timit/features/TRAIN/DR4/MMDM0/SI1941.npy
timit/features/TRAIN/DR4/MMDM0/SI1311.npy
timit/features/TRAIN/DR4/MMDM0/SX321.npy
timit/features/TRAIN/DR4/MCSS0/SX300.npy
timit/features/TRAIN/DR4/MCSS0/SX210.npy
timit/features/TRAIN/DR4/MCSS0/SI750.npy
timit/features/TRAIN/DR4/MCSS0/SI1380.npy
timit/features/TRAIN/DR4/MCSS0/SX390.npy
timit/features/TRAIN/DR4/MCSS0/SX30.npy
timit/features/TRAIN/DR4/MCSS0/SX120.npy
timit/features/TRAIN/DR4/MCDR0/SI524.npy
timit/features/TRAIN/DR4/MCDR0/SI1784.npy
timit/features/TRAIN/DR4/MCDR0/SI1154.npy
timit/features/TRAIN/DR4/MCDR0/SX74.npy
timit/features/TRAIN/DR4/MCDR0/SX344.npy
timit/features/TRAIN/DR4/MCDR0/SX434.npy
timit/features/TRAIN/DR4/MCDR0/SX254.npy
timit/features/TRAIN/DR4/MLEL0/SI1246.npy
timit/features/TRAIN/DR4/MLEL0/SI1876.npy
timit/featur

In [11]:
for split_name, split_path in {
    "train": core_train_path,
    "validation": core_val_path,
    "test": core_test_path,
}.items():
    df = pd.read_csv(TIMIT_ROOT / split_path)
    assert df["audio_path"].str.startswith(f"{config.name}/{config.raw_dir}/").all()
    assert df["feature_path"].str.startswith(f"{config.name}/{config.features_dir}/").all()
    assert not df["audio_path"].map(lambda path: sentence_id({"audio_path": path}).startswith("SA")).any()
    print(f"{split_name} dataset size: {len(df)}")

train dataset size: 3512
validation dataset size: 184
test dataset size: 192


In [12]:
df_train = pd.read_csv(TIMIT_ROOT / core_train_path)
df_val = pd.read_csv(TIMIT_ROOT / core_val_path)
df_test = pd.read_csv(TIMIT_ROOT / core_test_path)

assert set(df_train["audio_path"]).isdisjoint(set(df_val["audio_path"]))
assert set(df_train["audio_path"]).isdisjoint(set(df_test["audio_path"]))
print("split overlap checks passed")

split overlap checks passed


In [ ]:
# Generated layout:
# - timit/raw: original TIMIT files
# - timit/features: normalized MFCC + delta .npy files
# - timit/metadata: source file manifests
# - timit/splits: train/validation/test CSVs consumed by training
# Validation examples are held out from both training and normalization-statistics estimation.